In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/Delivery_Logistics.csv")
df.shape

(25000, 15)

In [3]:
drop_cols = [
    'delivery_id',
    'delivery_time_hours',
    'delivery_status',
    'delivery_rating'
]

df_model = df.drop(columns=drop_cols)

In [4]:
df_model.shape

(25000, 11)

In [5]:
X = df_model.drop(columns=['delayed'])
y = df_model['delayed']

In [6]:
print(X.shape)
print(y.shape)

(25000, 10)
(25000,)


In [7]:
y = y.map({
    'no': 0,
    'yes': 1
})

In [8]:
y.value_counts()

delayed
0    18331
1     6669
Name: count, dtype: int64

In [9]:
X.head()

,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,expected_time_hours,delivery_cost
0,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,1970-01-01 00:00:00.000000008,1632.7206
1,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,1970-01-01 00:00:00.000000003,640.1700
2,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,1970-01-01 00:00:00.000000016,1448.1700
3,dhl,electronics,ev van,same day,east,cold,269.7,12.69,1970-01-01 00:00:00.000000008,1486.5700
4,dhl,clothing,van,two day,north,foggy,256.7,37.02,1970-01-01 00:00:00.000000016,1394.5600


In [10]:
X['expected_time_hours'] = (
    X['expected_time_hours']
    .str.extract(r'(\d+)$')[0]
    .astype(int)
)

In [11]:
X['expected_time_hours'].head()

0     8
1     3
2    16
3     8
4    16
Name: expected_time_hours, dtype: int64

In [12]:
numeric_features = [
    'distance_km',
    'package_weight_kg',
    'expected_time_hours',
    'delivery_cost'
]

In [13]:
categorical_features = [
    'delivery_partner',
    'package_type',
    'vehicle_type',
    'delivery_mode',
    'region',
    'weather_condition'
]

In [14]:
print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

Numerical features:
['distance_km', 'package_weight_kg', 'expected_time_hours', 'delivery_cost']

Categorical features:
['delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode', 'region', 'weather_condition']


In [15]:
print("X columns:")
print(X.columns.tolist())

X columns:
['delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode', 'region', 'weather_condition', 'distance_km', 'package_weight_kg', 'expected_time_hours', 'delivery_cost']


In [16]:
from sklearn.model_selection import train_test_split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [18]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (20000, 10)
X_test: (5000, 10)
y_train: (20000,)
y_test: (5000,)


In [19]:
y_train.value_counts(normalize=True)

delayed
0    0.73325
1    0.26675
Name: proportion, dtype: float64

In [20]:
y_test.value_counts(normalize=True)

delayed
0    0.7332
1    0.2668
Name: proportion, dtype: float64

In [21]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

encoder = OneHotEncoder(handle_unknown='ignore')
scaler = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_features),
        ('num', scaler, numeric_features)
    ]
)

In [22]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [23]:
print("Original X_train:", X_train.shape)
print("Processed X_train:", X_train_processed.shape)

print("Original X_test:", X_test.shape)
print("Processed X_test:", X_test_processed.shape)

Original X_train: (20000, 10)
Processed X_train: (20000, 43)
Original X_test: (5000, 10)
Processed X_test: (5000, 43)


In [24]:
import joblib

joblib.dump(X_train_processed, "X_train_processed.pkl")
joblib.dump(X_test_processed, "X_test_processed.pkl")
joblib.dump(y_train, "y_train.pkl")
joblib.dump(y_test, "y_test.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

['preprocessor.pkl']

Logistic regression